# Lesson 07 - Obrasac dizajna planiranja

Ovaj bilježnik pokazuje **Obrazac dizajna planiranja** za AI agente koristeći Microsoft Agent Framework.
Naučit ćete kako razbiti složene zahtjeve za putovanje u strukturirane podzadatke, dodijeliti ih specijaliziranim agentima,
i izvršiti konačni plan — sve koristeći strukturirani izlaz potpomognut Pydantic modelima.


## Postavljanje


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Raspad zadatka

Raspad zadatka je srž obrasca dizajna planiranja. Umjesto da tražimo od jednog agenta da od početka do kraja obradi složen zahtjev, razbijamo problem na manje, jasno definirane **poddzadatke**.
Svaki podzadatak je dodijeljen specijaliziranom agentu (npr. letovi, hoteli, aktivnosti) s jasnim prioritetima i redoslijedom ovisnosti.

Ovaj pristup donosi nekoliko prednosti:
- **Jasnoća**: svaki podzadatak ima jednu odgovornost.
- **Paralelizam**: neovisni podzadatci mogu se izvršavati istovremeno.
- **Pouzdanost**: neuspjesi su izolirani na pojedinačne podzadatke.
- **Praćenje budžeta**: troškovi se procjenjuju po podzadatku i zbrajaju se.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Izrada agenta za planiranje sa strukturiranim izlazom

Agent za planiranje djeluje kao **koordinator na recepciji**. S obzirom na zahtjev za putovanje na visokoj razini, on proizvodi strukturirani `TravelPlan` — razlažući zahtjev na podzadatke, postavljajući prioritete i identificirajući ovisnosti kako bi konijerž ili sloj izvršenja mogli obaviti posao.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Izvršavanje plana s pomoćnim alatima

Kada je agent na recepciji izradio strukturirani plan, **concierge agent** ga izvršava.  
Svaki specijalistički alat obrađuje jednu kategoriju podzadatka (letovi, hoteli, aktivnosti). Concierge prolazi kroz podzadatke plana redoslijedom ovisnosti i upućuje svaki na odgovarajući alat.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Sažetak

U ovoj lekciji naučili ste **Obrazac planiranja** za AI agente:

1. **Razlaganje zadatka** — Agent na recepciji razlaže složen zahtjev za putovanje na
   strukturirane podzadatke koristeći Pydantic modele, dodjeljujući svaki specijaliziranom agentu s prioritetima
   i ovisnostima.
2. **Strukturirani izlaz** — Prosljeđivanjem `response_format` agent vraća validirani
   objekt `TravelPlan` umjesto slobodnog teksta, što čini daljnju obradu pouzdanom.
3. **Izvršenje plana** — Concierge agent prolazi kroz podzadatke koristeći specijalizirane alate
   (`book_flight`, `reserve_hotel`, `book_activity`) za provedbu plana i izvještavanje o rezultatima.

Ovaj obrazac odvaja *što raditi* (planiranje) od *kako to učiniti* (izvršenje), čineći agente
modularnijima, testabilnijima i lakšima za proširenje.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Odricanje od odgovornosti**:
Ovaj dokument preveden je pomoću AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati pogreške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati službenim izvorom. Za važne informacije preporučuje se stručni ljudski prijevod. Ne snosimo odgovornost za bilo kakva nesporazume ili pogrešne tumačenja koja proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
